# Data Exploration - Stock Prices and Financial News

This notebook explores the collected stock price data and financial news.

## Contents:
1. Load and examine stock price data
2. Basic statistics and distributions
3. Time series visualization
4. Correlation analysis
5. Missing data analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load Stock Price Data

In [ ]:
# Load stock price data
df_prices = pd.read_csv('../data/raw/stock_prices.csv', parse_dates=['Date'])

print(f"Dataset shape: {df_prices.shape}")
print(f"\nDate range: {df_prices['Date'].min()} to {df_prices['Date'].max()}")
print(f"\nTickers: {df_prices['Ticker'].unique()}")
print(f"\nFirst few rows:")
df_prices.head()

In [ ]:
# Data info
df_prices.info()

In [ ]:
# Basic statistics
df_prices.describe()

## 2. Time Series Visualization

In [ ]:
# Plot closing prices for all tickers
fig, ax = plt.subplots(figsize=(15, 7))

for ticker in df_prices['Ticker'].unique():
    ticker_data = df_prices[df_prices['Ticker'] == ticker]
    ax.plot(ticker_data['Date'], ticker_data['Close'], label=ticker, linewidth=2)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Close Price ($)', fontsize=12)
ax.set_title('Stock Closing Prices Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Normalized prices (to compare relative performance)
fig, ax = plt.subplots(figsize=(15, 7))

for ticker in df_prices['Ticker'].unique():
    ticker_data = df_prices[df_prices['Ticker'] == ticker].sort_values('Date')
    normalized = (ticker_data['Close'] / ticker_data['Close'].iloc[0]) * 100
    ax.plot(ticker_data['Date'], normalized, label=ticker, linewidth=2)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Price (Base 100)', fontsize=12)
ax.set_title('Normalized Stock Prices (Relative Performance)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axhline(y=100, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Volume Analysis

In [ ]:
# Average daily volume by ticker
avg_volume = df_prices.groupby('Ticker')['Volume'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
avg_volume.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Ticker', fontsize=12)
ax.set_ylabel('Average Daily Volume', fontsize=12)
ax.set_title('Average Trading Volume by Stock', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Average Daily Volume:")
print(avg_volume)

## 4. Price Distribution and Statistics

In [ ]:
# Daily returns distribution
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, ticker in enumerate(df_prices['Ticker'].unique()):
    ticker_data = df_prices[df_prices['Ticker'] == ticker].sort_values('Date')
    returns = ticker_data['Close'].pct_change() * 100
    
    ax = axes[idx]
    ax.hist(returns.dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Daily Returns (%)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(f'{ticker} Daily Returns Distribution', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Remove extra subplot if odd number of tickers
if len(df_prices['Ticker'].unique()) < 6:
    for idx in range(len(df_prices['Ticker'].unique()), 6):
        fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

In [ ]:
# Return statistics
return_stats = []

for ticker in df_prices['Ticker'].unique():
    ticker_data = df_prices[df_prices['Ticker'] == ticker].sort_values('Date')
    returns = ticker_data['Close'].pct_change() * 100
    
    return_stats.append({
        'Ticker': ticker,
        'Mean Return (%)': returns.mean(),
        'Std Dev (%)': returns.std(),
        'Min Return (%)': returns.min(),
        'Max Return (%)': returns.max(),
        'Sharpe Ratio': returns.mean() / returns.std() if returns.std() > 0 else 0
    })

returns_df = pd.DataFrame(return_stats)
print("\nDaily Returns Statistics:")
returns_df

## 5. Correlation Analysis

In [ ]:
# Create a pivot table for correlation analysis
price_pivot = df_prices.pivot(index='Date', columns='Ticker', values='Close')

# Calculate correlation
correlation_matrix = price_pivot.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Stock Price Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(correlation_matrix)

## 6. Missing Data Analysis

In [ ]:
# Check for missing values
missing_data = df_prices.isnull().sum()
missing_pct = (df_prices.isnull().sum() / len(df_prices)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Percentage (%)': missing_pct
})

print("Missing Data Summary:")
print(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print("\nNo missing values found!")

## 7. Key Insights

Based on the exploration above:

1. **Data Quality**: Check if there are any missing values or anomalies
2. **Volatility**: Which stocks show the most volatility?
3. **Correlation**: Are the stocks highly correlated? This matters for portfolio diversification
4. **Trends**: Are there clear upward or downward trends?
5. **Volume**: How liquid are these stocks?

## Next Steps

- Explore news sentiment data in notebook 02
- Add technical indicators
- Prepare features for modeling